# 186. 长期 Agent Memory：巩固、冲突更新与遗忘怎样实现？

> **面试问题：记忆不只是向量库：事实如何追加、修订、失效、检索、压缩，并避免继续使用过期偏好？**

## 先给结论

长期记忆应是带主体、时间、来源、置信、有效区间和权限的状态库。新观察先形成 append-only event，再由可验证 transition 做 add/revise/invalidate；检索必须偏好当前有效且有来源的记录。遗忘不是随便删除，而是停止使用已失效/过期/撤回信息，同时保留必要审计与隐私删除路径。

## 推荐回答主线

1. 定义 event 与 memory record，保留 valid_from/valid_to、source、tenant 和 supersedes 关系。
2. 巩固时校验覆盖、保真和冲突；更新产生新版本并失效旧版本，不原地抹掉历史。
3. 检索综合相关、时效、置信和权限，在 prompt budget 内选证据，并过滤过期事实。
4. 评估 recall、reasoning、recommendation、obsolete-use/FAMA、写错率、隐私和恢复能力。

## 教学实现边界

示例使用内存列表和词项打分，不调用向量数据库或 LLM 摘要器；重点是生命周期与时间语义。真实系统还需持久事务、加密、删除合规、并发冲突和人工修复。

## 一手资料

- [MemGPT](https://arxiv.org/abs/2310.08560)
- [Memora: From Recall to Forgetting](https://arxiv.org/abs/2604.20006)
- [TrustMem](https://arxiv.org/abs/2606.25161)


In [ ]:
import hashlib
import json
import math
import re
from dataclasses import asdict, dataclass, replace

import numpy as np

# 记录使用半开有效区间；tenant、subject 与 source 都是强制授权/溯源字段。
@dataclass(frozen=True)
class MemoryRecord:
    memory_id: str
    tenant: str
    subject: str
    key: str
    value: str
    valid_from: int
    valid_to: int | None
    source: str
    confidence: float
    supersedes: str | None = None

assert MemoryRecord.__dataclass_fields__["valid_to"] is not None
assert "subject" in MemoryRecord.__dataclass_fields__
assert "source" in MemoryRecord.__dataclass_fields__


## 1. Append-only event：原始观察与巩固事实分层

用户消息/工具结果先作为 immutable event，带 event id、主体、来源和时间。MemoryRecord 是从 event 推导的可查询状态；这样可重放新规则，也能追查错误记忆从哪来。


In [ ]:
@dataclass(frozen=True)
class MemoryEvent:
    event_id: str
    tenant: str
    subject: str
    observed_at: int
    source: str
    payload: str

def event_hash(event):
    return hashlib.sha256(json.dumps(asdict(event), sort_keys=True, ensure_ascii=False).encode()).hexdigest()

# 相同 event 摘要稳定，payload/tenant 变化可检测，事件保留主体与来源。
event = MemoryEvent("e1", "tenant-a", "user-7", 10, "user_message", "我现在住在上海")
assert event_hash(event) == event_hash(event)
assert event_hash(event) != event_hash(replace(event, payload="我现在住在北京"))
assert event.tenant and event.subject and event.source


## 2. 巩固 Add：只能写入有来源、合法时间和足够置信的事实

LLM 抽取结果不是事实。transition verifier 检查字段覆盖、payload 支持、租户/主体、置信与时间；不满足进入 quarantine，不能静默写长期库。


In [ ]:
def validate_event(event):
    fields = (event.event_id, event.tenant, event.subject, event.source, event.payload)
    if not all(isinstance(value, str) and value.strip() for value in fields):
        raise ValueError("event 必填字段不能为空")
    if not isinstance(event.observed_at, int) or event.observed_at < 0:
        raise ValueError("event time 非法")

def add_memory(event, memory_id, key, value, confidence):
    validate_event(event)
    if not all(isinstance(item, str) and item.strip() for item in (memory_id, key, value)):
        raise ValueError("memory id/key/value 不能为空")
    if not math.isfinite(confidence) or not 0.7 <= confidence <= 1.0:
        raise ValueError("置信度必须有限且位于 [0.7, 1]")
    if value not in event.payload:
        raise ValueError("value 缺少原始事件支持")
    return MemoryRecord(memory_id, event.tenant, event.subject, key, value, event.observed_at, None, event.event_id, confidence)

# 有原文支持的上海可写；幻觉、空值、NaN 与低置信都 fail closed。
memory_v1 = add_memory(event, "m1", "city", "上海", 0.95)
assert memory_v1.value == "上海" and memory_v1.source == "e1"
for bad_value, confidence in [("北京", 0.95), ("上海", 0.4), ("上海", float("nan")), ("", 0.95)]:
    try:
        add_memory(event, "bad", "city", bad_value, confidence); assert False
    except ValueError:
        assert True


## 3. Revise：新版本 supersede 旧版本，而非原地覆盖

用户搬家后，旧城市在历史时点仍为真，但当前已失效。修订创建新 record，并把旧 record 的 valid_to 设为新 valid_from；边界采用半开区间 `[from,to)`，避免同一时刻双有效。


In [ ]:
def revise_memory(old, new_event, new_id, new_value, confidence):
    if old.valid_to is not None:
        raise ValueError("只能修订当前有效版本")
    if old.tenant != new_event.tenant or old.subject != new_event.subject:
        raise ValueError("主体不一致")
    if new_event.observed_at <= old.valid_from:
        raise ValueError("修订事件必须晚于旧版本生效时间")
    new = replace(add_memory(new_event, new_id, old.key, new_value, confidence), supersedes=old.memory_id)
    closed_old = replace(old, valid_to=new.valid_from)
    return closed_old, new

# 修订创建新版本并闭合旧版本；回溯事件不能制造 valid_to < valid_from。
event2 = MemoryEvent("e2", "tenant-a", "user-7", 20, "user_message", "我已经搬到北京")
closed_v1, memory_v2 = revise_memory(memory_v1, event2, "m2", "北京", 0.98)
assert closed_v1.valid_to == memory_v2.valid_from == 20
assert memory_v2.supersedes == memory_v1.memory_id
try:
    revise_memory(memory_v2, replace(event2, event_id="e-back", observed_at=5), "m-back", "北京", 0.9); assert False
except ValueError:
    assert True


## 4. Canonical Store：事务化 Add→Revise→Invalidate

生产系统不能在 notebook 外部手工拼 `timeline`。下面的 store 先在候选副本上验证 event/memory id、来源、有效区间和同 key 重叠，再一次性提交；失败后 canonical state 保持逐字节不变。历史查询与在线查询都读取同一份版本链。


In [ ]:
def active_at(record, time):
    return record.valid_from <= time and (record.valid_to is None or time < record.valid_to)

def validate_records(events, records):
    groups = {}
    for record in records.values():
        if not all((record.memory_id, record.tenant, record.subject, record.key, record.value, record.source)):
            raise ValueError("record 必填字段为空")
        if not math.isfinite(record.confidence) or not 0 <= record.confidence <= 1:
            raise ValueError("record confidence 非法")
        if record.valid_to is not None and record.valid_to <= record.valid_from:
            raise ValueError("有效区间必须为非空半开区间")
        source = events.get(record.source)
        if source is None or (source.tenant, source.subject) != (record.tenant, record.subject):
            raise ValueError("record source 缺失或跨主体")
        groups.setdefault((record.tenant, record.subject, record.key), []).append(record)
    for versions in groups.values():
        versions.sort(key=lambda item: item.valid_from)
        for previous, current in zip(versions, versions[1:]):
            if previous.valid_to is None or current.valid_from < previous.valid_to:
                raise ValueError("同一事实版本区间重叠")

class MemoryStore:
    def __init__(self):
        self.events, self.records = {}, {}

    def snapshot(self):
        payload = {"events": {k: asdict(v) for k, v in sorted(self.events.items())},
                   "records": {k: asdict(v) for k, v in sorted(self.records.items())}}
        return json.dumps(payload, sort_keys=True, ensure_ascii=False)

    def _commit(self, candidate_events, candidate_records):
        validate_records(candidate_events, candidate_records)
        self.events, self.records = candidate_events, candidate_records

    def add(self, new_event, memory_id, key, value, confidence):
        if new_event.event_id in self.events or memory_id in self.records:
            raise ValueError("event/memory id 必须唯一")
        record = add_memory(new_event, memory_id, key, value, confidence)
        candidate_events = {**self.events, new_event.event_id: new_event}
        candidate_records = {**self.records, memory_id: record}
        self._commit(candidate_events, candidate_records)
        return record

    def revise(self, old_id, new_event, new_id, new_value, confidence):
        if old_id not in self.records or new_event.event_id in self.events or new_id in self.records:
            raise ValueError("修订 id 合同失败")
        closed, new = revise_memory(self.records[old_id], new_event, new_id, new_value, confidence)
        candidate_events = {**self.events, new_event.event_id: new_event}
        candidate_records = {**self.records, old_id: closed, new_id: new}
        self._commit(candidate_events, candidate_records)
        return closed, new

    def invalidate(self, memory_id, at_time):
        if memory_id not in self.records:
            raise ValueError("memory 不存在")
        record = self.records[memory_id]
        if not isinstance(at_time, int) or at_time <= record.valid_from:
            raise ValueError("失效时间必须晚于生效时间")
        closed = replace(record, valid_to=min(record.valid_to, at_time) if record.valid_to is not None else at_time)
        candidate_records = {**self.records, memory_id: closed}
        self._commit(dict(self.events), candidate_records)
        return closed

    @property
    def timeline(self):
        return sorted(self.records.values(), key=lambda item: (item.valid_from, item.memory_id))

def resolve_value(records, tenant, subject, key, time):
    candidates = [r for r in records if (r.tenant, r.subject, r.key) == (tenant, subject, key) and active_at(r, time)]
    if len(candidates) != 1:
        raise ValueError(f"需要唯一有效事实，实际 {len(candidates)}")
    return candidates[0]

# 重叠 add、重复 id 与回溯 revise 都事务失败；合法修订保留历史时点。
store = MemoryStore()
store.add(event, "m1", "city", "上海", 0.95)
before_duplicate = store.snapshot()
for duplicate_event, duplicate_memory in [
    (event, "m-new"),
    (MemoryEvent("e-new", "tenant-a", "user-7", 12, "user_message", "仍住在上海"), "m1"),
]:
    try:
        store.add(duplicate_event, duplicate_memory, "city", "上海", 0.9); assert False
    except ValueError:
        assert store.snapshot() == before_duplicate
before_overlap = store.snapshot()
overlap_event = MemoryEvent("e-overlap", "tenant-a", "user-7", 15, "user_message", "仍住在上海")
try:
    store.add(overlap_event, "m-overlap", "city", "上海", 0.9); assert False
except ValueError:
    assert store.snapshot() == before_overlap
closed_v1, memory_v2 = store.revise("m1", event2, "m2", "北京", 0.98)
assert resolve_value(store.timeline, "tenant-a", "user-7", "city", 15).value == "上海"
assert resolve_value(store.timeline, "tenant-a", "user-7", "city", 20).value == "北京"
before_backdated = store.snapshot()
try:
    store.revise("m2", MemoryEvent("e-old", "tenant-a", "user-7", 5, "user_message", "住在北京"), "m3", "北京", 0.9); assert False
except ValueError:
    assert store.snapshot() == before_backdated


## 5. Invalidate/Forget：失效必须写回 canonical store

用户撤回偏好时，逻辑失效应立即进入同一版本链，使在线检索停止使用；历史时点仍可审计。合规物理删除另走密钥擦除和备份流程，不能把“保留审计”和“继续喂给模型”混为一谈。


In [ ]:
# t=30 失效通过 store 原子提交：历史 t=29 可解析，在线 t>=30 不再 active。
forgotten_v2 = store.invalidate("m2", 30)
assert active_at(forgotten_v2, 29)
assert not active_at(forgotten_v2, 30)
assert resolve_value(store.timeline, "tenant-a", "user-7", "city", 29).value == "北京"
try:
    resolve_value(store.timeline, "tenant-a", "user-7", "city", 30); assert False
except ValueError:
    assert True
before_invalid = store.snapshot()
try:
    store.invalidate("m2", 19); assert False
except ValueError:
    assert store.snapshot() == before_invalid


## 6. 检索排序：Subject ACL 与时间过滤先于相似度

tenant 不是用户身份；同一 tenant 内仍可能有大量主体。在线检索先验证调用者允许访问请求 subject，再 hard filter tenant、subject 和有效区间，最后才按相关、时效与置信排序。负 limit 等非法预算必须拒绝，不能利用 Python 负切片绕过。


In [ ]:
def words(text):
    return set(re.findall(r"[a-z0-9]+|[\u4e00-\u9fff]+", text.casefold()))

def memory_score(query, record, now):
    overlap = len(words(query) & words(record.key + record.value)) / max(len(words(query)), 1)
    recency = math.exp(-0.05 * max(0, now - record.valid_from))
    return 0.5 * overlap + 0.2 * recency + 0.3 * record.confidence

def retrieve_memories(query, records, tenant, subject, allowed_subjects, now, limit):
    if not isinstance(limit, int) or limit < 0:
        raise ValueError("limit 必须是非负整数")
    if subject not in set(allowed_subjects):
        raise PermissionError("调用者没有该 subject 权限")
    eligible = [r for r in records if (r.tenant, r.subject) == (tenant, subject) and active_at(r, now)]
    return sorted(eligible, key=lambda r: (memory_score(query, r, now), r.memory_id), reverse=True)[:limit]

# 同租户另一主体即使更相关、更高置信，也不能污染 user-7 的真实检索主路径。
other_event = MemoryEvent("e-other", "tenant-a", "user-8", 24, "user_message", "城市城市城市北京")
other_record = add_memory(other_event, "m-other", "city", "北京", 1.0)
poisoned_timeline = store.timeline + [other_record]
retrieved = retrieve_memories("用户住在哪个城市", poisoned_timeline, "tenant-a", "user-7", {"user-7"}, 25, 3)
assert [record.memory_id for record in retrieved] == ["m2"]
assert retrieve_memories("城市", poisoned_timeline, "tenant-a", "user-7", {"user-7"}, 30, 3) == []
for bad_limit in (-1, 1.5):
    try:
        retrieve_memories("城市", poisoned_timeline, "tenant-a", "user-7", {"user-7"}, 25, bad_limit); assert False
    except ValueError:
        assert True
try:
    retrieve_memories("城市", poisoned_timeline, "tenant-a", "user-8", {"user-7"}, 25, 3); assert False
except PermissionError:
    assert True


## 7. Prompt 主路径：只压缩已授权、当前有效且可溯源的记录

prompt builder 自己调用受权检索，不能接收调用者随手拼好的候选列表。序列化保留 memory id、有效区间和 source；字符预算只作用于已经通过 subject/时间门禁的记录，因此压缩不会把过期或越权事实重新洗回上下文。


In [ ]:
def serialize_memory(record):
    end = record.valid_to if record.valid_to is not None else "open"
    return f"[{record.memory_id}|{record.valid_from},{end}|source={record.source}] {record.key}={record.value}"

def fit_memory_budget(records, char_budget):
    if not isinstance(char_budget, int) or char_budget < 0:
        raise ValueError("char_budget 必须是非负整数")
    selected, used = [], 0
    for record in records:
        serialized = serialize_memory(record)
        if used + len(serialized) <= char_budget:
            selected.append(serialized); used += len(serialized)
    return selected, used

def build_memory_prompt(query, records, tenant, subject, allowed_subjects, now, limit, char_budget):
    eligible = retrieve_memories(query, records, tenant, subject, allowed_subjects, now, limit)
    selected, used = fit_memory_budget(eligible, char_budget)
    return "\n".join(selected), {"eligible_ids": [r.memory_id for r in eligible], "used_chars": used}

# 历史 prompt 使用 m2 并保留来源；失效时点之后真实主入口返回空，不再使用 obsolete memory。
prompt_25, trace_25 = build_memory_prompt("城市", poisoned_timeline, "tenant-a", "user-7", {"user-7"}, 25, 3, 200)
prompt_30, trace_30 = build_memory_prompt("城市", poisoned_timeline, "tenant-a", "user-7", {"user-7"}, 30, 3, 200)
assert "m2" in prompt_25 and "source=e2" in prompt_25
assert trace_25["eligible_ids"] == ["m2"]
assert prompt_30 == "" and trace_30["eligible_ids"] == []
assert fit_memory_budget([memory_v2], 0) == ([], 0)


## 8. 遗忘感知评测与制品：惩罚使用 obsolete memory

只测“记住旧事实”会奖励错误持久化。报告 current/historical accuracy、obsolete-use rate、update latency、transition omission/corruption/hallucination、tenant leakage 与删除 SLA；FAMA 类指标显式惩罚过期记忆。


In [ ]:
def forgetting_aware_accuracy(correct, used_obsolete, obsolete_penalty=1.0):
    correct = np.asarray(correct, dtype=float)
    obsolete = np.asarray(used_obsolete, dtype=float)
    if correct.shape != obsolete.shape or correct.size == 0:
        raise ValueError("评测数组必须同形且非空")
    return float(np.mean(correct - obsolete_penalty * obsolete))

@dataclass(frozen=True)
class MemoryArtifact:
    schema: str
    consolidator: str
    transition_verifier: str
    retrieval_recipe: str
    retention_policy: str

# 使用过期记忆会降分；artifact 明确绑定 subject ACL、时间与 canonical transition。
assert forgetting_aware_accuracy([1, 1], [0, 0]) == 1.0
assert forgetting_aware_accuracy([1, 1], [0, 1]) < 1.0
artifact = MemoryArtifact("temporal-memory-v3", "extractor-v5", "transaction-transition-v4", "tenant-subject-acl-time-v5", "user-delete-v2")
digest = hashlib.sha256(json.dumps(asdict(artifact), sort_keys=True).encode()).hexdigest()
assert "subject-acl" in artifact.retrieval_recipe
assert digest != hashlib.sha256(json.dumps(asdict(replace(artifact, transition_verifier="transition-v5")), sort_keys=True).encode()).hexdigest()


## 面试收束：Agent/RAG 的算法只是控制面的一部分

推荐回答顺序是：任务目标和失败代价、状态/事件/证据合同、决策公式、可执行反例、离线与在线指标、权限和版本。受控环境只能证明状态机和数值关系，不能冒充开放网络、真实用户或真实模型结果。生产系统还要处理并发、超时、幂等、恶意内容、隐私、审计、灰度与回滚。

遇到追问时，主动区分模型判断与确定 verifier、计划与真实副作用、原始 observation 与 belief/memory、召回质量与生成归因，以及多尝试成功率与单次可靠性。
